# Programming Assignment: Numerical Data Transformation

Welcome to the Numerical Data Transformation Programming Assignment!

Numerical data rarely arrives ready to use. Real-world features span wildly different scales, follow skewed distributions, and mix continuous with count-based data. Mastering the right transformation for the right situation is one of the most valuable preprocessing skills — it directly determines what patterns your models can detect.

**What You Will Do in this Assignment:**

* **Scaling Comparison** - Apply StandardScaler, MinMaxScaler, and RobustScaler; compare their outputs on a dataset with outliers
* **Distribution Transforms** - Apply log, square root, Box-Cox, and Yeo-Johnson transforms and measure skewness reduction
* **Binning and Discretization** - Discretize continuous features using equal-width, equal-frequency, and custom domain-driven bins
* **Scaling Impact on KNN** - Measure and compare KNN accuracy on raw vs. scaled data to observe how distance-based models benefit from scaling
* **Transformation Pipeline** - Build a complete `ColumnTransformer + Pipeline` that handles skewed and regular numeric features
* **StandardScaler from Scratch** - Implement standardisation using only NumPy (mean, std, and the z-score formula)
* **MinMaxScaler from Scratch** - Implement min-max normalisation using only NumPy (min, range, and the normalisation formula)

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents

- [Exercise 1 – Scaling Comparison](#exercise-1)
- [Exercise 2 – Distribution Transformations](#exercise-2)
- [Exercise 3 – Binning and Discretization](#exercise-3)
- [Exercise 4 – Feature Scaling Impact on KNN](#exercise-4)
- [Exercise 5 – Transformation Pipeline](#exercise-5)
- [Exercise 6 – StandardScaler from Scratch](#exercise-6)
- [Exercise 7 – MinMaxScaler from Scratch](#exercise-7)

## Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests
from scipy import stats
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    FunctionTransformer,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Dataset

We use a synthetic e-commerce customer dataset with 500 customers:

| Column | Distribution | Description |
|:-------|:-------------|:------------|
| `income` | Log-normal (right-skewed) | Annual income |
| `age` | Uniform | Customer age |
| `price` | Normal with outliers | Last purchase price |
| `review_count` | Exponential | Number of reviews written |
| `premium` | Binary target | 1 if premium subscriber |

In [ ]:
np.random.seed(42)
N = 500

income       = np.random.lognormal(mean=10, sigma=1.2, size=N).round(2)
age          = np.random.randint(18, 70, N).astype(float)
price        = np.abs(np.random.normal(200, 80, N)).round(2)
review_count = (np.random.exponential(scale=10, size=N) + 1).astype(int).astype(float)

# Inject 5 outliers into price
price_w_outliers = price.copy()
price_w_outliers[:5] = [2000.0, 2500.0, 1800.0, 3000.0, 2200.0]

df = pd.DataFrame({'income': income, 'age': age,
                   'price': price_w_outliers, 'review_count': review_count})

logit = -2 + 0.5*(np.log(income)-10) + 0.02*age + np.random.normal(0, 1, N)
df['premium'] = (logit > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    df.drop('premium', axis=1), df['premium'], test_size=0.2, random_state=42)

print(f'Dataset shape: {df.shape}')
print(f'Premium rate: {df["premium"].mean():.2f}')
df.describe().round(2)

<a id='exercise-1'></a>
## Exercise 1 – Scaling Comparison

### Background

Different scalers suit different data characteristics:

| Scaler | Formula | Best for |
|:-------|:--------|:---------|
| **StandardScaler** | $z = (x - \mu) / \sigma$ | Approximate Gaussian, no heavy outliers |
| **MinMaxScaler** | $x' = (x - \min) / (\max - \min)$ | Bounded features, neural networks |
| **RobustScaler** | $x' = (x - \text{median}) / \text{IQR}$ | Data with real outliers |

> Always fit on **training data only**, then transform both train and test sets.

### Task

Implement `apply_scalers(X_train, X_test, columns)` that:
1. Applies **StandardScaler**, **MinMaxScaler**, and **RobustScaler** to the specified `columns`
2. Each scaler is **fit on `X_train` only**, then transforms both `X_train` and `X_test`
3. Returns a dict: `{'standard': (X_tr, X_te), 'minmax': (X_tr, X_te), 'robust': (X_tr, X_te)}`
4. The returned DataFrames hold the scaled column values; other columns are unchanged

**Hints:**
- `scaler.fit_transform(X_train[columns])` fits and transforms in one call.
- Use `scaler.transform(X_test[columns])` (no re-fitting!) for test data.
- Assign the scaled array back with `result[columns] = scaled_array`.

In [ ]:
def apply_scalers(X_train, X_test, columns):
    ### START CODE HERE ### (≈ 8 lines)
    results = {}
    for name, ScalerCls in [('standard', StandardScaler),
                             ('minmax',   MinMaxScaler),
                             ('robust',   RobustScaler)]:
        scaler = None
        X_tr_s = None   # fit on training, transform training
        X_te_s = None   # transform test (no re-fit)
        results[name] = (X_tr_s, X_te_s)
    ### END CODE HERE ###

In [ ]:
NUMERIC_COLS = ['income', 'age', 'price', 'review_count']
scaled_results = apply_scalers(X_train, X_test, NUMERIC_COLS)

for name, (X_tr, _) in scaled_results.items():
    print(f'{name:12s} | mean: {X_tr[NUMERIC_COLS].mean().round(3).tolist()} '
          f'| std: {X_tr[NUMERIC_COLS].std().round(3).tolist()}')

In [ ]:
unittests.exercise_1(apply_scalers)

<a id='exercise-2'></a>
## Exercise 2 – Distribution Transformations

### Background

Skewed distributions can hurt linear models and normality-sensitive algorithms. Four common transforms:

| Transform | Formula | Input requirement |
|:----------|:--------|:------------------|
| Log | $\ln(x)$ | Strictly positive |
| Square root | $\sqrt{x}$ | Non-negative |
| Box-Cox | Optimal $\lambda$ power | Strictly positive |
| Yeo-Johnson | Optimal $\lambda$ power | Any real value |

### Task

Implement `apply_distribution_transforms(series)` that:
1. Applies all four transforms to the input Series
2. Returns a dict `{'log': array, 'sqrt': array, 'boxcox': array, 'yeojohnson': array}`
   where each value is a NumPy array of the same length as the input

**Hints:**
- `np.log(series)` for the log transform.
- `np.sqrt(series)` for square root.
- `stats.boxcox(series)` returns `(transformed_array, lambda_value)` — take element 0.
- `stats.yeojohnson(series)` returns `(transformed_array, lambda_value)` — take element 0.

In [ ]:
def apply_distribution_transforms(series):
    ### START CODE HERE ### (≈ 6 lines)
    arr = series.values
    log_t   = None                    # np.log1p
    sqrt_t  = None                    # np.sqrt
    bc_t, _ = None                    # stats.boxcox  (returns array, lambda)
    yj_t, _ = None                    # stats.yeojohnson
    return {'log': log_t, 'sqrt': sqrt_t, 'boxcox': bc_t, 'yeojohnson': yj_t}
    ### END CODE HERE ###

In [ ]:
transforms = apply_distribution_transforms(df['income'])

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
all_data = [('Original', df['income'].values, 'Income')] + \
           [(k.capitalize(), v, k) for k, v in transforms.items()]
for ax, (title, data, _) in zip(axes, all_data):
    ax.hist(data, bins=30, alpha=0.8, edgecolor='white')
    ax.set_title(f'{title}\nSkew: {stats.skew(data):.2f}', fontsize=10)
    ax.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_2(apply_distribution_transforms)

<a id='exercise-3'></a>
## Exercise 3 – Binning and Discretization

### Background

Binning converts a continuous feature into discrete integer labels:

| Strategy | Method | When to use |
|:---------|:-------|:------------|
| Equal-width | `pd.cut` | Uniform scale; known physical ranges |
| Equal-frequency | `pd.qcut` | Skewed data; equal data representation |
| Custom edges | `pd.cut` with explicit edges | Domain-knowledge thresholds |

### Task

Implement `apply_binning(series, strategy, n_bins, custom_edges=None)` that:
1. Applies the chosen strategy: `'equal_width'`, `'equal_freq'`, or `'custom'`
2. For `'equal_width'`: uses `pd.cut(series, bins=n_bins, labels=False, include_lowest=True)`
3. For `'equal_freq'`: uses `pd.qcut(series, q=n_bins, labels=False, duplicates='drop')`
4. For `'custom'`: uses `pd.cut(series, bins=custom_edges, labels=False, include_lowest=True)`
5. Returns an integer-dtype `pd.Series` of bin indices (0-based)

**Hints:**
- `.astype(int)` to ensure integer output after `pd.cut` / `pd.qcut`.

In [ ]:
def apply_binning(series, strategy='equal_width', n_bins=5, custom_edges=None):
    ### START CODE HERE ### (≈ 6 lines)
    if strategy == 'equal_width':
        binned = None     # pd.cut
    elif strategy == 'equal_freq':
        binned = None     # pd.qcut
    else:                 # 'custom'
        binned = None     # pd.cut with custom_edges
    return binned
    ### END CODE HERE ###

In [ ]:
income_ew = apply_binning(df['income'], strategy='equal_width',   n_bins=5)
income_ef = apply_binning(df['income'], strategy='equal_freq',    n_bins=5)
age_custom = apply_binning(df['age'], strategy='custom',
                            custom_edges=[0, 25, 35, 50, 100])

print('Equal-width bin counts:',     income_ew.value_counts().sort_index().tolist())
print('Equal-freq bin counts:',      income_ef.value_counts().sort_index().tolist())
print('Custom age-group bin counts:', age_custom.value_counts().sort_index().tolist())

In [ ]:
unittests.exercise_3(apply_binning)

<a id='exercise-4'></a>
## Exercise 4 – Feature Scaling Impact on KNN

### Background

KNN computes **Euclidean distances** between points. When features have vastly different scales, the large-scale feature dominates the distance computation and the model effectively ignores the other features — even if those carry the true signal.

After `StandardScaler`, each feature contributes equally to the distance, allowing KNN to use all features.

### Task

Implement `compare_knn_scaling(X_train, y_train, X_test, y_test, k=5)` that:
1. Trains a `KNeighborsClassifier(n_neighbors=k)` on the **raw** features and evaluates accuracy
2. Applies `StandardScaler` (fit on `X_train`, transform both sets) and trains a second KNN
3. Returns `{'unscaled_accuracy': float, 'scaled_accuracy': float}`

**Hints:**
- Use `accuracy_score(y_test, model.predict(X_test))` for evaluation.
- `X_train` / `X_test` are NumPy arrays, not DataFrames.

In [ ]:
def compare_knn_scaling(X_train, y_train, X_test, y_test, k=5):
    ### START CODE HERE ### (≈ 8 lines)
    # unscaled
    knn_raw = None
    unscaled_acc = None
    # scaled
    scaler = None
    X_tr_s = None
    X_te_s = None
    knn_sc  = None
    scaled_acc = None
    return {'unscaled_accuracy': unscaled_acc, 'scaled_accuracy': scaled_acc}
    ### END CODE HERE ###

In [ ]:
# Build a dataset where one feature is signal and the other is high-scale noise
np.random.seed(42)
n_demo = 300
X_demo = np.column_stack([
    np.concatenate([np.random.randn(n_demo//2),       # class 0 feature 1
                    np.random.randn(n_demo//2) + 5]), # class 1 feature 1 (signal)
    np.concatenate([np.random.randn(n_demo//2)*800,   # class 0 feature 2 (noise)
                    np.random.randn(n_demo//2)*800]),  # class 1 feature 2 (noise)
])
y_demo = np.array([0]*(n_demo//2) + [1]*(n_demo//2))
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_demo, y_demo,
                                                    test_size=0.3, random_state=42)

result = compare_knn_scaling(X_tr_d, y_tr_d, X_te_d, y_te_d)
print(f'Unscaled KNN accuracy: {result["unscaled_accuracy"]:.3f}')
print(f'Scaled   KNN accuracy: {result["scaled_accuracy"]:.3f}')

In [ ]:
unittests.exercise_4(compare_knn_scaling)

<a id='exercise-5'></a>
## Exercise 5 – Transformation Pipeline

### Background

A production-ready numerical transformation pipeline applies different transforms to different columns inside a `ColumnTransformer`:

```
Input DataFrame
  ├─ log_cols  ──► log1p()  ──► StandardScaler ──┐
  └─ other_cols ──────────► StandardScaler ───────┼──► Estimator
```

Using `log1p` (instead of `log`) safely handles zeros. Wrapping everything in a `Pipeline` prevents leakage during cross-validation.

### Task

Implement `create_transformation_pipeline(numeric_cols, log_cols)` that:
1. For `log_cols`: creates a sub-`Pipeline` with `FunctionTransformer(np.log1p)` then `StandardScaler`
2. For columns in `numeric_cols` but **not** in `log_cols`: applies `StandardScaler` directly
3. Combines both via `ColumnTransformer`
4. Returns a `Pipeline([('preprocessing', ct), ('clf', LogisticRegression(max_iter=500))])`

**Hints:**
- `[c for c in numeric_cols if c not in log_cols]` gives the non-log columns.
- `FunctionTransformer(np.log1p)` wraps `np.log1p` as an sklearn transformer.

In [ ]:
def create_transformation_pipeline(numeric_cols, log_cols):
    ### START CODE HERE ### (≈ 7 lines)
    other_cols = None       # columns in numeric_cols but not in log_cols
    ct = ColumnTransformer([
        ('log',    None, log_cols),    # FunctionTransformer + StandardScaler
        ('std',    None, other_cols),  # StandardScaler only
    ])
    return Pipeline([('preprocessing', ct), ('clf', LogisticRegression(max_iter=1000))])
    ### END CODE HERE ###

In [ ]:
from sklearn.model_selection import cross_val_score

LOG_COLS = ['income', 'review_count']
ALL_COLS = ['income', 'age', 'price', 'review_count']

pipeline = create_transformation_pipeline(ALL_COLS, LOG_COLS)
X_pipe = df[ALL_COLS]
y_pipe = df['premium']

scores = cross_val_score(pipeline, X_pipe, y_pipe, cv=5, scoring='accuracy')
print(f'5-fold CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
unittests.exercise_5(create_transformation_pipeline)

<a id='exercise-6'></a>
## Exercise 6 – StandardScaler from Scratch

### Background

Z-score normalization centers and scales each feature:

$$z = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

The key rule: **$\mu$ and $\sigma$ are computed from the training set only**, then applied to both train and test.

### Task

Implement `standard_scaler_from_scratch(X_train, X_test)` where `X_train` and `X_test` are 2D NumPy arrays. The function must:
1. Compute the **column-wise mean** and **population standard deviation** (`ddof=0`) from `X_train`
2. Scale `X_train` and `X_test` using those statistics
3. Return `(X_train_scaled, X_test_scaled)` as `np.ndarray`

**Hints:**
- `X_train.mean(axis=0)` computes column means.
- `X_train.std(axis=0)` uses `ddof=0` (population std) by default — consistent with sklearn.

In [ ]:
def standard_scaler_from_scratch(X_train, X_test):
    ### START CODE HERE ### (≈ 5 lines)
    mean = None     # compute column-wise mean from X_train
    std  = None     # compute column-wise population std from X_train
    X_train_scaled = None   # apply z-score formula
    X_test_scaled  = None   # use training mean/std
    return X_train_scaled, X_test_scaled
    ### END CODE HERE ###

In [ ]:
X_np = df[['income', 'age', 'price', 'review_count']].values
X_tr_np = X_np[:400]
X_te_np = X_np[400:]

X_tr_sc, X_te_sc = standard_scaler_from_scratch(X_tr_np, X_te_np)
print(f'Train mean  (should be ≈ 0): {X_tr_sc.mean(axis=0).round(6).tolist()}')
print(f'Train std   (should be ≈ 1): {X_tr_sc.std(axis=0).round(6).tolist()}')
print(f'Test mean   (info only):      {X_te_sc.mean(axis=0).round(3).tolist()}')

In [ ]:
unittests.exercise_6(standard_scaler_from_scratch)

<a id='exercise-7'></a>
## Exercise 7 – MinMaxScaler from Scratch

### Background

Min-Max normalization maps each feature to the range $[0, 1]$:

$$x' = \frac{x - x_{\min,\text{train}}}{x_{\max,\text{train}} - x_{\min,\text{train}}}$$

Again, min and max are computed **from the training set only**. Test values may fall outside $[0, 1]$ — this is expected and correct.

### Task

Implement `minmax_scaler_from_scratch(X_train, X_test)` where inputs are 2D NumPy arrays. The function must:
1. Compute the **column-wise minimum and maximum** from `X_train`
2. Apply the formula to scale both `X_train` and `X_test`
3. Return `(X_train_scaled, X_test_scaled)` as `np.ndarray`

**Hints:**
- `X_train.min(axis=0)` and `X_train.max(axis=0)` for column-wise min/max.
- Compute `X_range = X_max - X_min` once and reuse it.

In [ ]:
def minmax_scaler_from_scratch(X_train, X_test):
    ### START CODE HERE ### (≈ 5 lines)
    X_min   = None   # column-wise min from X_train
    X_range = None   # column-wise (max - min) from X_train
    X_train_scaled = None   # apply min-max formula
    X_test_scaled  = None   # use training min/range
    return X_train_scaled, X_test_scaled
    ### END CODE HERE ###

In [ ]:
X_tr_mm, X_te_mm = minmax_scaler_from_scratch(X_tr_np, X_te_np)
print(f'Train min   (should be ≈ 0): {X_tr_mm.min(axis=0).round(6).tolist()}')
print(f'Train max   (should be ≈ 1): {X_tr_mm.max(axis=0).round(6).tolist()}')
print(f'Test  range (may exceed [0,1]): min={X_te_mm.min(axis=0).round(3).tolist()}, '
      f'max={X_te_mm.max(axis=0).round(3).tolist()}')

In [ ]:
unittests.exercise_7(minmax_scaler_from_scratch)